# Orchestration d’un débat : comparer sept modes par le budget

**Objectifs :** distinguer les registres d’arrêt, comparer les axes compatibles et calculer une projection. **Prérequis :** LINQ et collections C#. **Durée :** 45 minutes.

Ces données sont les 14 observations historiques du 15 septembre 2026 : même modèle et corpus, budgets 180 et 600 secondes. Elles proviennent de `orchestration_modes.py` et de l’instrument EPITA `scripts/compare_orchestration_modes.py`. Coûts observés : 0,5504 et 0,9485 USD. Aucun fournisseur n’est exécuté ici. Le huitième mode `conversation_real` est connu mais non mesuré dans ce snapshot.

**Série Argumentation** | Précédent : [07b-Communication](./Argumentation-07b-Communication-Channels-CSharp.ipynb) | Suivant : [08-Capstone](./Argumentation-08-Capstone-Python.ipynb) | [↑ Série Argumentation](README.md)

**Les calculs C# sont déterministes** : stdlib pure, aucune dependance, aucun aleatoire.
Les 14 observations transcrites (7 modes x 2 budgets) portent chacun leur perimetre
de travail (`scope_of_work`) ; la table d'asymetrie structurelle (`DEPTH_PARITY`)
porte ses constantes mesurees avec leur provenance. Le port documente ses
divergences en docstring du module — les runs LLM ne sont pas portes (les
mesures le sont), l'introspection vivante des workflows est remplacee par les
constantes mesurees, et la plage de delegation degeneree « 4-4 » est conservee
comme distribution, pas comme entier.
Le fil de lecture du carnet suit la structure de la source : un tableau
sous-budget qui a l'air juste (§1), la preuve que ses colonnes ne mesurent
pas la meme chose (§2), le meme tableau a budget calibre qui le renverse
(§3), et la regle qui derive un budget d'une mesure — avec la limite
casee de sa forme naive (§4). Chaque section lit son output **apres**
l'avoir produit : les valeurs citees dans la prose sont celles des cellules
precedentes, pas des souvenirs du carnet source.
**Perimetre du snapshot (signale par audit #18435, mesure du 2026-09-30).** Les 14 observations transcrites (7 modes x 2 budgets) datent du 2026-09-15. Depuis, l'instrument du tronc en balaie **huit** : la livraison #2456 du 2026-09-23 a ajoute `conversation_real` (builder dedie, kernel dedie, runner dedie, scope : "ConversationOrchestrator(mode=real, informal + FOL agents, LLM + Tweety)"). Ce huitieme mode est **annonce** dans le texte comme mode connu non mesuré mais ne figure pas dans les observations (les mesures n'ont pas ete re-integrees).
Consequence pour le lecteur : les exercices et les tableaux qui suivent (verdicts, bascules du filet, projections lineaires, derivation de budget) restent exacts sur leur perimetre de 7 modes ; aucune valeur citee n'est fausse -- c'est l'exhaustivite qui est perimee. Le mode manquant est precisement celui qui complete la lecon du §2 (non-comparabilite des axes) : meme DAG conversationnel que `conversation_deterministic`, mais avec de vrais agents informels et FOL, LLM + Tweety -- le seul mode conversationnel qui exerce le raisonnement formel.


In [1]:
// Les lignes suivantes sont des mesures historiques, non des résultats de fournisseur rejoués.
using System;
using System.Linq;
using System.Collections.Generic;

// Une mesure garde les trois registres d'arrêt et la portée réellement exercée.
record Run(string Mode, bool Success, double Duration, int Done, int Total, bool Killed, bool Exhaustive, double? Fill, string Scope, bool Terminates = true, bool Decides = true)
{
    public double PhasesRatio => Total == 0 ? 0 : (double)Done / Total;
}

record Depth(string Mode, string Dimension, int Count, string Nature, string Range);
var under = new Run[]
{
    new Run("pipeline_standard", false, 180.01, 6, 15, true, false, 0.196, "UnifiedPipeline DAG workflow"),
    new Run("pipeline_light", true, 148.76, 3, 3, false, true, 0.179, "UnifiedPipeline DAG workflow"),
    new Run("pipeline_full", false, 180.02, 7, 17, true, false, 0.196, "UnifiedPipeline DAG workflow"),
    new Run("conversational", true, 180.01, 1, 3, false, false, 0.096, "AgentGroupChat multi-agent (plafond interne 180 s)"),
    new Run("conversation_deterministic", true, 0.061, 3, 3, false, false, null, "ConversationOrchestrator(mode=demo, SimulatedAgent, no LLM)"),
    new Run("hierarchical_bridge", false, 180.01, 2, 4, true, false, 0.118, "Strategic planning -> objectives_to_workflow (4 axes)"),
    new Run("hierarchical_delegation", false, 180.02, 1, 5, true, false, null, "Strategic -> Tactical -> Operational (3-tier, 5 tasks)")
};
var calibrated = new Run[]
{
    new Run("pipeline_standard", true, 461.36, 15, 15, false, true, 0.462, "UnifiedPipeline DAG workflow"),
    new Run("pipeline_light", true, 178.8, 3, 3, false, true, 0.179, "UnifiedPipeline DAG workflow"),
    new Run("pipeline_full", true, 448.12, 17, 17, false, true, 0.487, "UnifiedPipeline DAG workflow"),
    new Run("conversational", true, 600.02, 2, 3, false, false, 0.308, "AgentGroupChat multi-agent (plafond interne 600 s)"),
    new Run("conversation_deterministic", true, 0.058, 3, 3, false, false, null, "ConversationOrchestrator(mode=demo, SimulatedAgent, no LLM)"),
    new Run("hierarchical_bridge", true, 236.38, 4, 4, false, false, null, "Strategic planning -> objectives_to_workflow (4 axes)"),
    new Run("hierarchical_delegation", true, 217.02, 3, 5, false, false, null, "Strategic -> Tactical -> Operational (3-tier, 5 tasks)")
};
var depth = new Depth[]
{
    new Depth("pipeline_light", "workflow phases (DAG)", 3, "breadth", null),
    new Depth("pipeline_standard", "workflow phases (DAG)", 15, "breadth", null),
    new Depth("pipeline_full", "workflow phases (DAG)", 17, "breadth", null),
    new Depth("hierarchical_bridge", "strategic objectives (default axes)", 4, "delegation", null),
    new Depth("hierarchical_delegation", "strategic objectives (LLM-derived, measured)", 4, "delegation (3-tier depth)", "4-4 objectifs -> 5-5 taches (n=3, corpus_A/B/C, firsthand R711, po-2023 projet-is)"),
    new Depth("conversational", "dialogue macro-phases (multi-turn)", 3, "dialogue-depth", null),
    new Depth("conversation_deterministic", "dialogue macro-phases (deterministic)", 3, "dialogue-depth (no LLM)", null)
};
// Le filet prime sur une déclaration de succès émise avant l’arrêt.
string Verdict(Run r) => r.Killed ? "tué par le filet du harnais" : r.Success && r.Done == r.Total ? "phases complètes, mode satisfait" : r.Success ? "mode satisfait d’un travail partiel" : "inachevé (auto-déclaration d’échec)";
string Family(Depth d) => d.Nature.Split(" (")[0];
string DepthVerdict(IEnumerable<Depth> ds) => $"{ds.Count()} modes, {ds.Select(Family).Distinct().Count()} familles";
double? Project(Run r) => r.Done <= 0 ? null : r.Duration * r.Total / r.Done;
// Le maximum ne porte que sur les mesures interrompues et projetables.
int? Budget(IEnumerable<Run> rs, double margin = .2)
{
    var ps = rs.Where(r => r.Killed).Select(Project).Where(x => x.HasValue).Select(x => x.Value).ToArray();
    return ps.Length == 0 ? null : (int)Math.Ceiling(ps.Max() * (1 + margin));
}

Console.WriteLine("Instrument : scripts/compare_orchestration_modes.py (EPITA #1735) ; modèle : gpt-5.6-luna BYOK ; corpus_A synthétique ; mesure du 2026-09-15");
Console.WriteLine("Budgets historiques : 180 s ($0.5504), 600 s ($0.9485)");
Console.WriteLine($"Observations historiques : {under.Length + calibrated.Length} ; conversation_real non mesuré.");

Instrument : scripts/compare_orchestration_modes.py (EPITA #1735) ; modèle : gpt-5.6-luna BYOK ; corpus_A synthétique ; mesure du 2026-09-15


Budgets historiques : 180 s ($0.5504), 600 s ($0.9485)


Observations historiques : 14 ; conversation_real non mesuré.


Les deux runs ne different **que** par le budget : meme modele, meme corpus,
memes modes. C'est la condition du contraste — si une autre variable bougeait,
la difference entre les deux tableaux ne prouverait rien. Le premier run
(sous-budget, 180 s) est celui qu'un experimentaliste lance « pour voir » ;
le second (calibre, 600 s) est celui que la mesure du premier a **derive**.

## 1. Le run sous-budget — le tableau qui a l'air juste

Chaque mode recoit le meme budget. Trois colonnes du tableau disent trois
choses differentes, et c'est leur ecart qui est le sujet.

In [2]:
void Table(IEnumerable<Run> records)
{
    foreach (var r in records)
        Console.WriteLine($"{r.Mode, -28} {Verdict(r), -35} {r.Duration, 7:F2} s {r.Done}/{r.Total} filet={r.Killed} exhaustif={r.Exhaustive} remplissage={r.Fill} périmètre={r.Scope}");
}

Table(under);
Console.WriteLine($"Filet : {under.Count(r => r.Killed)}/{under.Length}");

pipeline_standard            tué par le filet du harnais          180.01 s 6/15 filet=True exhaustif=False remplissage=0.196 périmètre=UnifiedPipeline DAG workflow


pipeline_light               phases complètes, mode satisfait     148.76 s 3/3 filet=False exhaustif=True remplissage=0.179 périmètre=UnifiedPipeline DAG workflow


pipeline_full                tué par le filet du harnais          180.02 s 7/17 filet=True exhaustif=False remplissage=0.196 périmètre=UnifiedPipeline DAG workflow


conversational               mode satisfait d’un travail partiel  180.01 s 1/3 filet=False exhaustif=False remplissage=0.096 périmètre=AgentGroupChat multi-agent (plafond interne 180 s)


conversation_deterministic   phases complètes, mode satisfait       0.06 s 3/3 filet=False exhaustif=False remplissage= périmètre=ConversationOrchestrator(mode=demo, SimulatedAgent, no LLM)


hierarchical_bridge          tué par le filet du harnais          180.01 s 2/4 filet=True exhaustif=False remplissage=0.118 périmètre=Strategic planning -> objectives_to_workflow (4 axes)


hierarchical_delegation      tué par le filet du harnais          180.02 s 1/5 filet=True exhaustif=False remplissage= périmètre=Strategic -> Tactical -> Operational (3-tier, 5 tasks)


Filet : 4/7


**Lecture du resultat.** Quatre modes sur sept finissent tues par le filet du
harnais : les deux pipelines longs (`standard` a 6/15 phases, `full` a 7/17)
et les deux hierarchiques (`bridge` a 2/4, `delegation` a 1/5) — soit quatre
arrêts. Le pont hiérarchique s'arrête à mi-parcours et la délégation
hiérarchique après une tâche sur cinq. Les trois survivants
racontent autre chose : `pipeline_light` termine ses 3 phases en 148,76 s ;
`conversation_deterministic` finit en **0,061 s** dans les données historiques
(0,06 s après arrondi dans le tableau, aucun LLM, dialogue simulé) ; et `conversational`... arrete sa course a exactement 180,01 s sans
etre tue. Les trois registres du tableau suivant expliquent pourquoi ce
dernier point n'est pas un detail.

Un lecteur pressé lirait ce tableau comme un podium : trois « gagnants »,
quatre « perdants ». C'est exactement la lecture que la source qualifie
d'artefact — les quatre perdants ne doivent leur place qu'a un parametre
externe à leur architecture. La colonne `remplissage` indique le taux
de remplissage de l’état partagé. Ses valeurs renseignées vont de 0,096 à 0,196.
La comparaison avec les observations calibrées plus bas montre le
remplissage du `pipeline_standard` passer de 0,196 à 0,462.

In [3]:
foreach (var r in under.Where(x => x.Mode == "pipeline_standard" || x.Mode == "conversational"))
    Console.WriteLine($"{r.Mode}: filet={r.Killed}, déclaration={r.Success}, phases={r.Done}/{r.Total}, exhaustif={r.Exhaustive}, périmètre={r.Scope}");

pipeline_standard: filet=True, déclaration=False, phases=6/15, exhaustif=False, périmètre=UnifiedPipeline DAG workflow


conversational: filet=False, déclaration=True, phases=1/3, exhaustif=False, périmètre=AgentGroupChat multi-agent (plafond interne 180 s)


**Lecture du resultat.** `pipeline_standard` : le filet du harnais a tire
(`oui`), le mode ne se declare pas satisfait, la realite dit 6 phases sur 15 —
les trois registres sont coherents, l'echec est net. `conversational` : le
filet dit `non` — ce n'est **pas** le harnais qui l'a arrete — pourtant sa
duree affiche 180,01 s, au budget au centieme pres. C'est son **plafond
interne** : le mode est borne en wall-time par construction (il s'arrete de
lui-meme a 180 s), et il declare `success` avec 1 phase sur 3. La distinction
a une consequence pratique : relever le budget du harnais ne changera **rien**
pour un mode plafonne en interne — la colonne « Filet » du tableau ne mesure
pas la meme chose que la colonne « Duree ».

## 2. Pourquoi le classement est faux — les modes ne partagent pas le meme axe

Un mode « plus long » n'est pas un mode « plus faible » : les sept modes ne
mesurent pas la meme chose. L'introspection ci-dessous est **deterministe**
(aucun LLM, aucun run) : elle lit la structure des modes, pas leurs
performances.

### Exercice 1 — Compter les verdicts

Regroupez les modes du run sous-budget par verdict, puis comptez chaque catégorie. Vérifiez que leur somme couvre les sept modes.

In [4]:
// Exercice 1 : compter les verdicts sous-budget.
var counts = new Dictionary<string, int>(); // TODO étudiant
Console.WriteLine(counts.Count == 0 ? "Exercice à compléter" : string.Join(", ", counts));

Exercice à compléter


## 2. Comparer les axes structurels

Les phases des pipelines, les niveaux de délégation et les tours de conversation ne mesurent pas la même profondeur. Examinons leurs unités avant de les comparer.

In [5]:
foreach (var d in depth)
    Console.WriteLine($"{d.Mode, -28} {d.Dimension, -40} {d.Count} {d.Nature} {d.Range}");
Console.WriteLine(string.Join(", ", depth.Select(Family).Distinct()));

pipeline_light               workflow phases (DAG)                    3 breadth 


pipeline_standard            workflow phases (DAG)                    15 breadth 


pipeline_full                workflow phases (DAG)                    17 breadth 


hierarchical_bridge          strategic objectives (default axes)      4 delegation 


hierarchical_delegation      strategic objectives (LLM-derived, measured) 4 delegation (3-tier depth) 4-4 objectifs -> 5-5 taches (n=3, corpus_A/B/C, firsthand R711, po-2023 projet-is)


conversational               dialogue macro-phases (multi-turn)       3 dialogue-depth 


conversation_deterministic   dialogue macro-phases (deterministic)    3 dialogue-depth (no LLM) 


breadth, delegation, dialogue-depth


**Lecture du resultat.** Sept modes, trois familles d'axes : la **largeur**
(breadth — le catalogue de phases DAG du pipeline : 3, 15, 17 phases), la
**delegation** (peu d'objectifs, plusieurs etages — 4 axes pour le bridge,
palier 3-tier pour la delegation), la **profondeur de dialogue** (3
macro-phases multi-tours). Les comptes ne sont **pas** sur une echelle
commune : 17 phases de DAG et 3 macro-phases de dialogue disent des choses
differentes sur des structures differentes — c'est exactement pourquoi la
colonne « Axe » existe. Noter la ligne `hierarchical_delegation` : son compte
porte une **plage mesuree** (« 4-4 objectifs -> 5-5 taches, n=3 ») et non un
entier — le palier est derive par LLM dans le moteur d'origine, donc son
compte est une distribution mesuree (qui se trouve d'etre degeneree : trois
corpus, toujours 4 objectifs). Le facteur 100 du tableau du §1 se rejoue ici :
`pipeline_full` (17 phases, 448 s calibre) et `conversation_deterministic`
(3 macro-phases, 0,06 s) sont tous deux « termines » — comparer leurs durees
n'a pas de sens, et l'instrument source l'ecrit noir sur blanc.

La colonne « Axe » est la reponse methodologique a cette non-comparabilite :
plutot que de fabriquer une echelle commune (aligner les comptes en gonflant
ou en tronquant), la table **etiquette** ce que chaque compte mesure. Un
lecteur qui voudrait quand meme ranger les modes par profondeur devrait au
minimum ranger **par famille** — 17 > 15 > 3 phases a l'interieur de la
famille breadth a un sens ; 17 phases contre 3 macro-phases n'en a aucun.
C'est la difference entre une metrique et une etiquette de metrique : la
premiere se compare, la seconde se lit.

In [6]:
Console.WriteLine(DepthVerdict(depth));
Console.WriteLine("Pipelines seuls : " + DepthVerdict(depth.Take(3)));

7 modes, 3 familles


Pipelines seuls : 3 modes, 1 familles


**Lecture du resultat.** Le verdict compte « 7 modes » et « 3 axes » parce que
la table en contient sept et trois — pas parce qu'une phrase en dure le
souvenir. La seconde commande le demontre : restreinte aux trois pipelines,
la meme fonction derive « 3 modes » et « 1 axe » sans qu'aucun litteral ne
soit mis a jour. C'est une discipline de fiabilite du prose (un litteral
obsoleterait silencieusement au premier ajout d'un mode — famille d'incidents
#1019 sur le moteur source), et elle coute une ligne : deriver les comptes de
la structure decrite. L'asymetrie elle-meme est un **choix de conception
documente** : aligner les axes exigerait soit tronquer le catalogue du
pipeline, soit gonfler artificiellement la profondeur des autres — les deux
fabriqueraient une parite factice.

## 3. Le run calibre — memes modes, meme texte, budget derive de la mesure

Le budget calibre n'est pas choisi au confort. La source documente qu'a
180 s, `pipeline_standard` meurt a 6/15 de ses phases, et qu'il lui faut
environ 500 s pour atteindre 15/15 sur un corpus court. Le run calibre prend
donc 600 s — au-dessus du besoin documente, avec une marge. C'est la regle
que ce carnet enseigne : **calibrer l'instrument avant de comparer**.

In [7]:
Table(calibrated);
Console.WriteLine($"Filet={calibrated.Count(r => r.Killed)} ; succès={calibrated.Count(r => r.Success)} ; complets={calibrated.Count(r => r.Done == r.Total)}");

pipeline_standard            phases complètes, mode satisfait     461.36 s 15/15 filet=False exhaustif=True remplissage=0.462 périmètre=UnifiedPipeline DAG workflow


pipeline_light               phases complètes, mode satisfait     178.80 s 3/3 filet=False exhaustif=True remplissage=0.179 périmètre=UnifiedPipeline DAG workflow


pipeline_full                phases complètes, mode satisfait     448.12 s 17/17 filet=False exhaustif=True remplissage=0.487 périmètre=UnifiedPipeline DAG workflow


conversational               mode satisfait d’un travail partiel  600.02 s 2/3 filet=False exhaustif=False remplissage=0.308 périmètre=AgentGroupChat multi-agent (plafond interne 600 s)


conversation_deterministic   phases complètes, mode satisfait       0.06 s 3/3 filet=False exhaustif=False remplissage= périmètre=ConversationOrchestrator(mode=demo, SimulatedAgent, no LLM)


hierarchical_bridge          phases complètes, mode satisfait     236.38 s 4/4 filet=False exhaustif=False remplissage= périmètre=Strategic planning -> objectives_to_workflow (4 axes)


hierarchical_delegation      mode satisfait d’un travail partiel  217.02 s 3/5 filet=False exhaustif=False remplissage= périmètre=Strategic -> Tactical -> Operational (3-tier, 5 tasks)


Filet=0 ; succès=7 ; complets=5


**Lecture du resultat.** Zero tue, sept succes auto-declares — mais le
tableau n'est pas « tout vert » pour autant : seuls **5 modes sur 7** ont
completement epuise leurs phases. `conversational` s'arrete a 2/3 (son
plafond interne a suivi le budget : 600,02 s, toujours sans filet), et
`hierarchical_delegation` declare succes a **3 phases sur 5**. Ce dernier
est le cas-ecole du §1 : le mode est satisfait d'un travail partiel — son
verdict (`mode satisfait d'un travail partiel`) n'est ni un echec ni un
complet, c'est un troisieme etat que seul le registre « realite » rend
visible. Les trois pipelines, eux, completent (15/15, 3/3, 17/17) et
marquent leur perimetre exhaustif. Le classement du §1 est renverse : les
quatre modes « faibles » d'hier terminent tous aujourd'hui.

Reste ce que le budget ne peut pas acheter : `perimeter_is_exhaustive` est
passe a `True` pour les trois pipelines seulement. Meme a budget ouvert, les
modes conversationnels et hierarchiques ne pretendent pas epuiser leur
perimetre — le plafond interne de l'un et la satisfaction partielle de
l'autre sont des proprietes de **conception**, pas des contraintes de temps.
Un budget infini ne changerait ni le 2/3 du conversational ni le 3/5 du
delegation : ces deux modes sont construits pour s'arreter en declarant
l'essentiel fait. C'est la frontiere entre ce qu'un experimentaliste peut
corriger (le filet) et ce qu'il doit documenter (le contrat du mode).

### Exercice 2 — Comparer les phases par seconde

Pour le run calibré, divisez les phases accomplies par la durée et classez les valeurs au sein de chaque famille d'axes. Interprétez ce que ce classement mesure. Le coût en dollars a été relevé pour le run entier.

In [8]:
// Exercice 2 : comparer phases/seconde uniquement au sein d’une même famille.
var efficiency = new List<(string Mode, double Value)>(); // TODO étudiant
Console.WriteLine(efficiency.Count == 0 ? "Exercice à compléter" : string.Join(", ", efficiency));

Exercice à compléter


## 3. Isoler les bascules liées au budget

Apparions les deux observations de chaque mode. Une bascule correspond ici à un arrêt par le filet dans le premier run et à son absence dans le run calibré.

In [9]:
foreach (var u in under)
{
    var c = calibrated.Single(r => r.Mode == u.Mode);
    Console.WriteLine($"{u.Mode, -28} bascule={u.Killed && !c.Killed} {u.Duration:F2} -> {c.Duration:F2} s");
}

Console.WriteLine($"Bascules : {under.Count(u => u.Killed && !calibrated.Single(c => c.Mode == u.Mode).Killed)}");

pipeline_standard            bascule=True 180.01 -> 461.36 s


pipeline_light               bascule=False 148.76 -> 178.80 s


pipeline_full                bascule=True 180.02 -> 448.12 s


conversational               bascule=False 180.01 -> 600.02 s


conversation_deterministic   bascule=False 0.06 -> 0.06 s


hierarchical_bridge          bascule=True 180.01 -> 236.38 s


hierarchical_delegation      bascule=True 180.02 -> 217.02 s


Bascules : 4


**Lecture du resultat.** Quatre bascules exactement — les quatre modes tues
a 180 s passent tous sous le filet a 600 s, avec des durees reelles qui
s'etalent de 217 a 461 s : personne n'avait besoin de 600 s sauf le mode
plafonne en interne. `conversational` et `conversation_deterministic` ne
basculent pas, pour deux raisons differentes : le premier ne doit son arret
qu'a son propre plafond, le second n'a jamais ete en danger (0,06 s). C'est
la demonstration complete : le « classement » du run sous-budget etait une
photographie du filet, pas une propriete des architectures. Deux runs
identiques a une variable pres suffisent a l'etablir — c'est la definition
meme d'un contraste experimental propre.

## 4. Deriver le budget — la regle, et la limite de sa forme naive

La regle de calibration dit : derive le budget de la **mesure**, pas du
confort. Sa forme la plus simple est une projection lineaire : un mode tue
apres `d` secondes en ayant fait `k` phases sur `n` — son travail complet
devrait couter environ `d * n / k`. Voyons ce que cette regle rend sur le
run sous-budget, et ce que le run calibre en dit.

Les bascules sont calculées par jointure sur le nom du mode.

## 4. Projection

In [10]:
foreach (var r in under.Where(r => r.Killed).OrderBy(Project))
{
    var c = calibrated.Single(x => x.Mode == r.Mode);
    Console.WriteLine($"{r.Mode, -28} {r.Done}/{r.Total} -> {Project(r):F1} s ; observé {c.Duration:F2} s");
}

hierarchical_bridge          2/4 -> 360.0 s ; observé 236.38 s


pipeline_full                7/17 -> 437.2 s ; observé 448.12 s


pipeline_standard            6/15 -> 450.0 s ; observé 461.36 s


hierarchical_delegation      1/5 -> 900.1 s ; observé 217.02 s


**Lecture du resultat.** Les projections ne se trompent pas dans le meme
sens pour tous les modes. Les deux hierarchiques sont **massivement
surestimes** : `hierarchical_delegation` projete a 900,1 s pour un reel a
217,0 s (x4,1), `hierarchical_bridge` a 360,0 s pour 236,4 s (x1,5). Les
deux pipelines sont au contraire **legerement sous-estimes** :
`pipeline_full` a 437,2 s pour 448,1 s, `pipeline_standard` a 450,0 s pour
461,4 s — environ 2,5 % sous le reel chacun.

L'ecart des hierarchiques est compatible avec un cout fixe concentre en
tete de run (mise en route, chargement du contexte, etablissement des
agents) : une phase deja faite y coute en moyenne **plus** que les
suivantes, et l'extrapolation lineaire herite ce biais sur tout le restant.
L'ecart des pipelines, lui, ne se lit pas : le reel vient d'un **second**
run, et le §3 montre qu'un mode complet dans les deux runs varie bien plus
d'un run a l'autre (`pipeline_light` : 148,76 s puis 178,80 s, +20 %).
La projection n'est donc ni une borne superieure ni une borne inferieure :
c'est un ordre de grandeur, juste a quelques pour cent pres sur deux modes
et faux d'un facteur 4 sur un autre.

In [11]:
var budget = Budget(under);
Console.WriteLine($"Budget naïf={budget} s ; budget historique=600 s ; ratio={(double)budget / 600:F2}");
var longestComplete = calibrated.Where(r => r.Done == r.Total).MaxBy(r => r.Duration);
if (longestComplete != null)
    Console.WriteLine($"Mode complet le plus long : {longestComplete.Mode} ({longestComplete.Duration:F2} s)");
foreach (var r in calibrated.Where(r => r.Success && r.Done < r.Total))
    Console.WriteLine($"Partiel : {r.Mode} {r.Done}/{r.Total}");

Budget naïf=1081 s ; budget historique=600 s ; ratio=1.80


Mode complet le plus long : pipeline_standard (461.36 s)


Partiel : conversational 2/3


Partiel : hierarchical_delegation 3/5


**Lecture du resultat.** La derivation naive rend **1081 s** (la projection
la plus longue, 900,1 s pour le mode tue a 1 phase sur 5, majoree de 20 %) ;
le run calibre reel a suffi a **600 s** — surestimation x1,8. La regle
correcte est donc plus subtile que « extrapoler le plus lent » : la source a
retenu le besoin **documente du mode pivot** (`pipeline_standard`, environ
500 s mesures au prealable) plus une marge, soit 600 s — et ce budget a
suffi a faire terminer tout le monde sauf les deux modes qui s'arretent
d'eux-memes. Les deux dernieres lignes le disent : le mode complet le plus
long, `pipeline_standard`, a eu besoin de 461,4 s ; `conversational` tourne
600,0 s, mais c'est son **propre** plafond interne qui l'arrete a 2 phases
sur 3 — sa duree mesure le plafond, pas le besoin. La lecon pour
l'experimentaliste : la plus longue projection, majoree, donne un
**plafond** defendable (1081 s couvre largement les 461,4 s du besoin
reel), pas une cible ;
et le budget choisi (600 s) a ete valide **par la mesure du run suivant** —
zero tue — pas par la projection seule. L'instrument et la mesure se
corrigent mutuellement, en deux temps.

Cette boucle en deux temps (mesurer, calibrer, re-mesurer) est le vrai
produit de l'instrument — plus que tout classement particulier. Un
experimentaliste qui saute le premier temps compare des timeouts ; un qui
s'arrete au deuxieme ne sait pas si son budget genereux ne masque pas un mode
plafonne en interne. La source l'a paye en dollars reels (0,55 puis 0,95) :
ici la boucle se rejoue en lecture de donnees committes, pour le cout d'une
execution de cellules.

### Exercice 3 : projeter un pipeline fictif incomplet à 300 secondes.

Complétez ce squelette puis comparez votre résultat au contrat.

In [12]:
// Exercice 3 : projeter un pipeline fictif incomplet à 300 secondes.
double? prediction = null; // TODO étudiant : construire un Run puis appeler Project
Console.WriteLine(prediction?.ToString() ?? "Exercice à compléter");

Exercice à compléter


## A retenir

1. **Un budget par defaut n'est pas un budget neutre.** Il fabrique un
   classement (« qui meurt le plus tard ») qui ressemble a un resultat
   d'architecture. Les deux runs de ce carnet ne different que par cette
   variable — et leurs tableaux disent l'inverse l'un de l'autre.
2. **Calibrer avant de comparer.** Le budget calibre est derive de la mesure
   precedente, pas choisi au confort — et la derivation naive (projection
   lineaire du plus lent) surestime d'un facteur 1,8 : les couts fixes des
   premieres phases gonflent l'extrapolation.
3. **Trois registres d'arret, pas un.** Filet du harnais, declaration du
   mode, realite des phases : ils divergent reellement (un mode peut se
   declarer satisfait a 3 phases sur 5 ; un autre s'arreter tout seul au
   budget sans jamais toucher le filet).
4. **Les modes ne partagent pas le meme axe.** Largeur, delegation,
   profondeur de dialogue : comparer des durees entre axes distincts n'a pas
   de sens — le facteur 100 entre le pipeline plein et le dialogue simule
   n'est pas un resultat, c'est une non-comparabilite.

**Position dans la serie** : les carnets precedents construisent ce qu'on
arbitre — `Toulmin_Model` deploie un argument, `Schemes_Walton` etiquette sa
forme, `Knowledge_Base` en garde la memoire, `Dung_AF_Semantics` tranche les
attaques. Celui-ci regarde **au-dessus** : quand plusieurs architectures
d'agents produisent le meme type de verdict, laquelle choisir ? La reponse
de l'instrument est methodologique — un axe de comparaison honnete par
famille, un budget calibre, et des registres d'arret qui ne mentent pas
chacun dans leur registre.

## Vérifiez votre compréhension

1. Pourquoi le budget par défaut peut-il fabriquer un classement trompeur ?
2. Pourquoi une projection linéaire peut-elle surestimer les modes hiérarchiques ?
3. Pourquoi le budget naïf calculé dépasse-t-il le budget du second run ?
4. Pourquoi comparer directement le pipeline complet au dialogue simulé serait-il trompeur ?

<details><summary>Éléments de réponse</summary>

1. Le filet arrête certains modes avant qu'ils aient terminé.
2. Un coût fixe en début de run peut gonfler le coût moyen des premières phases ; le bridge et la délégation sont surestimés de facteurs différents.
3. Il majore la plus longue projection, qui suppose un coût uniforme des phases.
4. Ils mesurent des axes de travail distincts.
</details>

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place de ce carnet dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet.

- Le rung [7-Orchestration](./Argumentation-07-Orchestration-Python.ipynb) pour les deux paradigmes que ce carnet arbitre par le budget.
- Le rung [7b-Communication-Channels](./Argumentation-07b-Communication-Channels-Python.ipynb) pour ce qui relie les agents d'une architecture.
- Le rung [8b-Executor](./Argumentation-08b-Executor-Python.ipynb) pour le budget de tours mesuré cette fois **par phase**, sur le pipeline de production.
- Le tronc (`2025-Epita-Intelligence-Symbolique`) pour les sept architectures d'agents et les campagnes d'exécution dont les mesures sont embarquées ici.

**Référence** — A. Georges, D. Buytaert et L. Eeckhout (2007). « Statistically Rigorous Java Performance Evaluation », Proceedings of OOPSLA 2007 — coûts fixes de mise en route, échauffement et variance run-to-run dans la mesure de performance.
